In [ ]:
#크롬드라이버 다운로드
%pip install webdriver-manager
%conda install requests beautifulsoup4 lxml selenium pandas

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# 드라이버 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get('https://example.com/')

html = driver.page_source       # 정적에서 response.text와 동일
soup = BeautifulSoup(html,'lxml')
result = soup.select_one('body>div>p:nth-child(2)')
print(result.text)

#종료
driver.quit()

# 자동차 판매량

In [ ]:
url = 'https://auto.danawa.com/auto/?Work=record'

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# 드라이버 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# 페이지 열기
driver.get(url)

html = driver.page_source       # 정적에서 response.text와 동일
soup = BeautifulSoup(html,'lxml')
result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div > div:nth-child(3) > div.left > table > tbody')
result = result.select('tr')
for tag in result:
    print(tag.select_one('.num').text, tag.select_one('a').text.strip())

#종료
driver.quit()

# 다나와 2025년 1월부터 12월까지 국산자동차 판매대수 수집
https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month=2025-01-00

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup
import time

company, amount = [],[]
year,month = [],[]

 # 드라이버 설정
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

for _month in range(1,13):
    _year = '2025'
    _url = f'https://auto.danawa.com/auto/?Work=record&Tab=Grand&Month={_year}-{_month:02}-00'   
    

    # 페이지 열기
    driver.get(_url)
    time.sleep(1)  # 페이지 완전 로딩될때까지 기다림.. 대략 1초면 충분하다고 판단

    html = driver.page_source   # 정적에서 response.text 와 동일
    soup = BeautifulSoup(html, 'lxml')
    result = soup.select_one('#autodanawa_gridC > div.gridMain > article > main > div:nth-child(1) > table > tbody')
    result = result.select('tr')
    count = len(result)
    for tag in result:        
        # print(tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] )
        co, am =  tag.select_one('a').text.strip(), tag.select_one('.num').contents[0] 
        company.append(co); amount.append(am)
    year += [_year]*count
    month += [_month]*count
    
    #종료
    driver.quit()

In [ ]:
import pandas as pd
data = {
    'year' : year,
    'month' : month,
    'company' : company,
    'amount' : amount
}
df = pd.DataFrame(data)
df

# mysql 데이터베이스 적재하기

In [ ]:
import mysql.connector
from mysql.connector import Error
from dotenv import load_dotenv
import os
# 환경변수 읽어오기
load_dotenv()

# 환경변수에서 MySQL 연결 정보 로드
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'user': os.getenv('DB_USER', 'root'),
    'password': os.getenv('DB_PASSWORD', ''),
    'database': os.getenv('DB_NAME', 'car_db'),
    'port': int(os.getenv('DB_PORT', 3306))
}

conn = mysql.connector.connect(**DB_CONFIG)
cursor = conn.cursor()
query = '''
insert into sale_car('year','month','compay','amount') 
values(%s,%s,%s,%s)
'''

for _, row in df.iterrows():
    params = (row.year, row.month,row.company, row.amount)
    cursor.execute(query,params=params)

cursor.close()
conn.close()

In [ ]:
for _, row in df.iterrows():
    print(row.year, row.month,row.company, row.amount)